In [1]:
import os

os.chdir("..")  # move up one level to the project root
print("New working directory:", os.getcwd())

New working directory: c:\Users\DELL\projects\Analytics\olist_analytics-data_warehouse


In [2]:
from src.utils.run_bronze import run_sql_folder

In [3]:
# Run Bronze queries
run_sql_folder("sql/bronze")

Running sql/bronze\bronze_customer.sql...


Running sql/bronze\bronze_geolocation.sql...
Running sql/bronze\bronze_order.sql...
Running sql/bronze\bronze_order_items.sql...
Running sql/bronze\bronze_order_payments.sql...
Running sql/bronze\bronze_order_reviews.sql...
Running sql/bronze\bronze_prod_cat_name.sql...
Running sql/bronze\bronze_products.sql...
Running sql/bronze\bronze_sellers.sql...


In [4]:
import duckdb



conn = duckdb.connect("olist.duckdb")

tables = conn.execute("SHOW TABLES").fetchall()
print("Tables in DuckDB:", tables)

conn.close()

Tables in DuckDB: [('bronze_customers',), ('bronze_geolocation',), ('bronze_order_items',), ('bronze_order_payments',), ('bronze_order_reviews',), ('bronze_orders',), ('bronze_prod_cat_name',), ('bronze_products',), ('bronze_sellers',), ('silver_customers',)]


In [5]:
import duckdb
import pandas as pd

conn = duckdb.connect("olist.duckdb")

df = conn.execute("SELECT * FROM bronze_customers").fetchdf()
orders_df = conn.execute("SELECT * FROM bronze_orders").fetchdf()
cat_name = conn.execute("SELECT * FROM bronze_prod_cat_name").fetchdf()
customers_df = pd.DataFrame(df)
orders_df = pd.DataFrame(orders_df)
cat_name_df = pd.DataFrame(cat_name)
conn.close()


In [6]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [7]:
customers_df

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,03937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,06764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [8]:
customers_df.describe().T

,count,unique,top,freq
customer_id,99441,99441,06b8999e2fba1a1fbc88172c00ba8bc7,1
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17
customer_zip_code_prefix,99441,14994,22790,142
customer_city,99441,4119,sao paulo,15540
customer_state,99441,27,SP,41746


In [9]:
duplicates = customers_df[customers_df.duplicated()]
duplicates

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [10]:
customers_df["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [11]:
orders_df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


In [12]:
orders_df["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [14]:
orders_df[orders_df.duplicated()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [15]:
orders_df[orders_df["order_approved_at"] < orders_df["order_purchase_timestamp"]]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [16]:
orders_df[orders_df["order_delivered_customer_date"] < orders_df["order_purchase_timestamp"]]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [17]:
orders_df[orders_df["order_delivered_customer_date"] < orders_df["order_purchase_timestamp"]]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [18]:
import duckdb
import os

print("Current folder:", os.getcwd())

conn = duckdb.connect("olist.duckdb")

# Make sure the CSV path is correct
csv_path = os.path.join(
    os.getcwd(),
    "data",
    "raw",
    "olist_orders_dataset.csv"
)

print("CSV exists:", os.path.exists(csv_path))

# Create table
conn.execute(f"""
CREATE TABLE IF NOT EXISTS bronze_orders AS
SELECT *
FROM read_csv_auto('{csv_path}')
""")

# Check tables
tables = conn.execute("SHOW TABLES").fetchall()
print("Tables in DuckDB:", tables)

conn.close()

Current folder: c:\Users\DELL\projects\Analytics\olist_analytics-data_warehouse
CSV exists: True
Tables in DuckDB: [('bronze_customers',), ('bronze_geolocation',), ('bronze_order_items',), ('bronze_order_payments',), ('bronze_order_reviews',), ('bronze_orders',), ('bronze_prod_cat_name',), ('bronze_products',), ('bronze_sellers',), ('silver_customers',)]


In [19]:
run_sql_folder("sql/silver")

Running sql/silver\silver_customers.sql...
Running sql/silver\silver_geolocation.sql...
Running sql/silver\silver_orders.sql...


In [20]:
import duckdb
import pandas as pd

conn = duckdb.connect("olist.duckdb")

df = conn.execute("SELECT * FROM silver_customers").fetchdf()

silver_customer = pd.DataFrame(df)

conn.close()

In [21]:
silver_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [22]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [23]:
customers_df[customers_df.duplicated()].sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: str

In [24]:
customers_df[customers_df.duplicated(subset=['customer_unique_id'], keep=False)]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
13,eabebad39a88bb6f5b52376faec28612,295c05e81917928d76245e842748184d,05704,sao paulo,SP
32,2d5831cb2dff7cdefba62e950ae3dc7b,e9dd12dca17352644a959d9dea133935,42800,camacari,BA
33,b2bed119388167a954382cca36c4777f,e079b18794454de9d2be5c12b4392294,27525,resende,RJ
...,...,...,...,...,...
99324,5b46a0d983eec8c97363bea78d4a69dd,8bab3162259edfaadd1ea2e1fe7f58dc,31565,belo horizonte,MG
99327,c1affa46f9f3b514555259049a0307b9,12ab9334b1240d6d037f2b0102a49571,38050,uberaba,MG
99336,ebf46ff530343a129926adc1f831dea4,0ee57f62666561b72f2ceacad0230cbf,09530,sao caetano do sul,SP
99353,282fbce48e4d2077aad602dd125c9225,0ceb502fc33a2ad327b08288c5310e2e,29134,viana,ES


In [25]:
silver_customer[silver_customer.duplicated(subset=['customer_unique_id'], keep=False)]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
13,eabebad39a88bb6f5b52376faec28612,295c05e81917928d76245e842748184d,05704,sao paulo,SP
32,2d5831cb2dff7cdefba62e950ae3dc7b,e9dd12dca17352644a959d9dea133935,42800,camacari,BA
33,b2bed119388167a954382cca36c4777f,e079b18794454de9d2be5c12b4392294,27525,resende,RJ
...,...,...,...,...,...
99324,5b46a0d983eec8c97363bea78d4a69dd,8bab3162259edfaadd1ea2e1fe7f58dc,31565,belo horizonte,MG
99327,c1affa46f9f3b514555259049a0307b9,12ab9334b1240d6d037f2b0102a49571,38050,uberaba,MG
99336,ebf46ff530343a129926adc1f831dea4,0ee57f62666561b72f2ceacad0230cbf,09530,sao caetano do sul,SP
99353,282fbce48e4d2077aad602dd125c9225,0ceb502fc33a2ad327b08288c5310e2e,29134,viana,ES


In [26]:
import duckdb

conn = duckdb.connect("olist.duckdb")  # or wherever your .duckdb file lives

with open("sql/quality_checks/silver/customer_checks.sql", "r") as f:
    sql_script = f.read()

# Split on semicolons to run each query separately and print results
queries = [q.strip() for q in sql_script.split(";") if q.strip() and not q.strip().startswith("--")]

for query in queries:
    print(f"\n--- Running ---\n{query}\n")
    result = conn.execute(query).fetchdf()
    print(result)

conn.close()